# Demo: Calabi--Yau Orientifolds and F-theory Uplifts

This notebook demonstrates the basic workflow provided by the `CY_orientifold` and `F_Theory_Uplift` classes in **CYTools**.

We will:

1. define a toric Calabi--Yau hypersurface,
2. choose a $\mathbb{Z}_2$ orientifold action,
3. construct the corresponding orientifold,
4. build its F-theory uplift, and
5. perform a small scan for nef partitions.

The helper routines are collected in `Uplift_functions.py`, while the main classes are implemented in `FT_CY.py`.

## 1. Setup

Basic imports

In [1]:
from cytools import Polytope, fetch_polytopes
from cytools.vector_config import VectorConfiguration

from cytools.f_theory import FT_CY as FT
from cytools.f_theory import Uplift_functions as UF

## 2. Example model: $\mathbb{P}_{[1,1,1,1,4]}$

We begin with the weighted-projective model $\mathbb{P}_{[1,1,1,1,4]}$. Its polytope may be defined either directly from its vertices or from its GLSM weights using `UF.points_from_glsm`.

This is a trilayer model. Before constructing the orientifold, we place the polytope in trilayer normal form.

In [2]:
# Direct definition from the vertices.
p_from_vertices = Polytope(
    [
        [-1,  0,  0,  0],
        [ 1,  1,  0,  0],
        [ 1,  0,  1,  0],
        [ 1,  0,  0,  1],
        [ 1, -1, -1, -1],
    ]
)

# Equivalent definition from the GLSM weights.
p = Polytope(UF.points_from_glsm([[1, 1, 1, 1, 4]]))

print("Is trilayer:", p.is_trilayer())
print("Vertices before normalization:")
print(p.vertices())

Is trilayer: True
Vertices before normalization:
[[-1 -1 -1 -4]
 [ 0  0  0  1]
 [ 0  0  1  0]
 [ 0  1  0  0]
 [ 1  0  0  0]]


In [3]:
p = UF.trilayer_normal_form(p)

print("Vertices in trilayer normal form:")
print(p.vertices())

Vertices in trilayer normal form:
[[-1  0  0  0]
 [ 1 -1 -1 -3]
 [ 1  0  0  1]
 [ 1  0  1  1]
 [ 1  1  0  1]]


## 3. Constructing the orientifold

A $\mathbb{Z}_2$ action is encoded by a vector $\xi$.

For a trilayer model, `CY_orientifold` automatically chooses the trilayer vertex $v_*$ when no action is supplied. Alternatively, the inequivalent $\mathbb{Z}_2$ actions can be computed explicitly from the automorphism group of the polytope.

In [4]:
# Automatic choice of the Z2 action for a trilayer model.
orientifold_trilayer = FT.CY_orientifold(p)

# Explicit list of inequivalent Z2 actions.
z2_actions = UF.inequivalent_Z2_actions(
    p.automorphisms(action="left")
)
print(z2_actions)

[[0 0 0 1]
 [1 0 0 0]
 [0 1 0 0]
 [1 1 0 1]
 [1 1 0 0]]


### Accepted input data

The `CY_orientifold` class accepts the following input formats:

- a list or array of points together with a $\mathbb{Z}_2$ action;
- a `Polytope` together with a $\mathbb{Z}_2$ action, unless the polytope is trilayer;
- a `VectorConfiguration` together with a $\mathbb{Z}_2$ action; or
- a triangulation or `Fan`, which is used as the ambient triangulation, together with a $\mathbb{Z}_2$ action.

The next cell constructs orientifold objects from a vector configuration and from one of its triangulations.

In [5]:
points = p.points_not_interior_to_facets()[1:]
xi = z2_actions[1]

vector_configuration = VectorConfiguration(points)
triangulation = vector_configuration.triangulate()

orientifold_from_vc = FT.CY_orientifold(
    vector_configuration,
    xi,
)
orientifold_from_triangulation = FT.CY_orientifold(
    triangulation,
    xi,
)

### Cartier/nef refinement

With `construct_nef_decomposition=True`, initialization attempts to construct the maximal refined normal fan needed for the nef decomposition. This option is enabled by default.

If the refined normal fan is not admissible, the construction falls back to the original toric data.

In [6]:
orientifold = FT.CY_orientifold(
    p,
    construct_nef_decomposition=True,
)

## 4. Basic orientifold data

The orientifold object provides access to the divisor class $D_B$, the ambient toric fan, the rays of the orbifold fan, and the normal fan.

When no triangulation is supplied and the uplift admits a nef decomposition, a triangulation is chosen automatically so that the relevant divisors are Cartier and nef.

In [7]:
print("Line bundle D_B:")
print(orientifold.line_bundle())

print("\nToric orbifold fan:")
print(orientifold.orbifold_toric_fan())

print("\nOrbifold rays:")
print(orientifold.vectors_orbifold())

print("\nNormal fan:")
print(orientifold.normal_fan())

Line bundle D_B:
[1 0 0 0 0 0]

Toric orbifold fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #6 vectors

Orbifold rays:
[[-1  0  0  0]
 [ 2 -1 -1 -3]
 [ 2  0  1  1]
 [ 2  1  0  1]
 [ 2  0  0  1]
 [ 1  0  0  0]]

Normal fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #5 vectors


## 5. F-theory uplift

An `F_Theory_Uplift` object accepts the same geometric input data as a `CY_orientifold`. When an orientifold object is supplied, it is used directly.

The following cell checks whether the construction gives a nef partition, the associated polytopes, and computes the Hodge numbers.

In [8]:
uplift = FT.F_Theory_Uplift(orientifold)

print("Is nef partition:", uplift.is_nef_partition())

print("\nM-lattice polytopes:")
print("  Base:       ", uplift.pol_B_M())
print("  Weierstrass:", uplift.pol_W_M())
print("  Sum:        ", uplift.pol_M_sum())
print("  Convex hull:", uplift.pol_M_conv())

print("\nN-lattice polytopes:")
print("  Base:       ", uplift.pol_B_N())
print("  Weierstrass:", uplift.pol_W_N())
print("  Sum:        ", uplift.pol_N_sum())
print("  Convex hull:", uplift.pol_N_conv())

print("\nHodge numbers:")
print("  h11 =", uplift.h11())
print("  h31 =", uplift.h31())

Is nef partition: True

M-lattice polytopes:
  Base:        A 4-dimensional lattice polytope in ZZ^6 
  Weierstrass: A 6-dimensional lattice polytope in ZZ^6 
  Sum:         A 6-dimensional reflexive lattice polytope in ZZ^6 
  Convex hull: A 6-dimensional reflexive lattice polytope in ZZ^6 

N-lattice polytopes:
  Base:        A 1-dimensional lattice polytope in ZZ^6 
  Weierstrass: A 6-dimensional lattice polytope in ZZ^6 
  Sum:         A 6-dimensional reflexive lattice polytope in ZZ^6 
  Convex hull: A 6-dimensional reflexive lattice polytope in ZZ^6 

Hodge numbers:
  h11 = 2
  h31 = 3878


### Toric fans of the uplift

The uplift keeps track of three ambient toric fans:

- the orientifold/orbifold fan $\widetilde{V}_4$;
- the ambient varietty of the uplift before resolving non-Higgsable clusters; and
- the ambient variety of the uplift after the required resolutions.

In [9]:
print("Orbifold toric fan:")
print(uplift.orbifold_toric_fan())

print("\nSingular uplift ambient fan:")
print(uplift.singular_uplift_ambient_toric_fan())

print("\nSmooth uplift ambient fan:")
print(uplift.smooth_uplift_ambient_toric_fan())

Orbifold toric fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #6 vectors

Singular uplift ambient fan:
A fine triangulation of a 6-dimensional vector configuration consisting of #9 vectors

Smooth uplift ambient fan:
A fine triangulation of a 6-dimensional vector configuration consisting of #11 vectors


## 6. Line bundles and intersection numbers

The base and Weierstrass line bundles are available in both the $M$-lattice and $N$-lattice descriptions.

In [10]:
print("M-lattice line bundles:")
print("  Base:       ", uplift.line_bundle_base_M())
print("  Weierstrass:", uplift.line_bundle_weierstrass_M())

print("\nN-lattice line bundles:")
print("  Base:       ", uplift.line_bundle_base_N())
print("  Weierstrass:", uplift.line_bundle_weierstrass_N())

M-lattice line bundles:
  Base:        [0 0 0 ... 1 1 1]
  Weierstrass: [1 1 1 ... 0 0 0]

N-lattice line bundles:
  Base:        [1 0 0 0 0 0 0 0 0 0 0]
  Weierstrass: [0 1 1 1 1 1 1 1 1 1 1]


The next command computes the intersection numbers of the ambient toric sixfold of the smooth uplift. The result is returned as a dictionary, and vanishing intersection numbers are omitted.

In [11]:
intersection_numbers = (
    uplift.intersection_numbers_smooth_uplift_ambient()
)
print(intersection_numbers)

{(1, 2, 3, 4, 7, 8): 1.0, (1, 2, 3, 4, 7, 9): 0.3333333333, (1, 2, 3, 4, 8, 9): 0.5, (1, 2, 3, 5, 7, 8): 1.0, (1, 2, 3, 5, 7, 9): 0.3333333333, (1, 2, 3, 5, 8, 9): 0.5, (1, 2, 4, 5, 7, 8): 1.0, (1, 2, 4, 5, 7, 9): 0.3333333333, (1, 2, 4, 5, 8, 9): 0.5, (1, 3, 4, 5, 7, 8): 1.0, (1, 3, 4, 5, 7, 9): 0.3333333333, (1, 3, 4, 5, 8, 9): 0.5, (2, 3, 4, 6, 7, 9): 0.3333333333, (2, 3, 4, 6, 7, 11): 0.3333333333, (2, 3, 4, 6, 8, 9): 0.5, (2, 3, 4, 6, 8, 10): 1.0, (2, 3, 4, 6, 10, 11): 1.0, (2, 3, 4, 7, 8, 10): 1.0, (2, 3, 4, 7, 10, 11): 1.0, (2, 3, 5, 6, 7, 9): 0.3333333333, (2, 3, 5, 6, 7, 11): 0.3333333333, (2, 3, 5, 6, 8, 9): 0.5, (2, 3, 5, 6, 8, 10): 1.0, (2, 3, 5, 6, 10, 11): 1.0, (2, 3, 5, 7, 8, 10): 1.0, (2, 3, 5, 7, 10, 11): 1.0, (2, 4, 5, 6, 7, 9): 0.3333333333, (2, 4, 5, 6, 7, 11): 0.3333333333, (2, 4, 5, 6, 8, 9): 0.5, (2, 4, 5, 6, 8, 10): 1.0, (2, 4, 5, 6, 10, 11): 1.0, (2, 4, 5, 7, 8, 10): 1.0, (2, 4, 5, 7, 10, 11): 1.0, (3, 4, 5, 6, 7, 9): 0.3333333333, (3, 4, 5, 6, 7, 11): 0.333333

## 7. A small scan at $h^{1,1}=2$

As a final example, we scan the first 100 polytopes with $h^{1,1}=2$ and retain the orientifold actions whose F-theory uplifts define nef partitions.

This is intentionally a small demonstration scan. Larger scans should include progress logging, exception handling, and persistent output files.

> Depending on the machine, this cell may take roughly 20--30 seconds.

In [12]:
nef_partitions = []

for polytope in fetch_polytopes(h11=2, limit=100):
    automorphisms = polytope.automorphisms(action="left")
    actions = UF.inequivalent_Z2_actions(automorphisms)

    for action in actions:
        candidate = FT.F_Theory_Uplift(
            polytope,
            action,
            construct_nef_decomposition=True,
        )
        if candidate.is_nef_partition():
            nef_partitions.append(candidate)

In [13]:
print(f"Number of nef partitions found: {len(nef_partitions)}")

Number of nef partitions found: 72


### Convenience iterators

The package also provides convenience functions for iterating directly over orientifolds and F-theory uplifts. They accept the same filtering data as `fetch_polytopes`; in particular, the Hodge numbers refer to the original Calabi--Yau hypersurface.

The optional filters `only_nef_decomposition=True` and `only_nef_partition=True` restrict the returned objects accordingly.

In [14]:
print("Orientifolds:")
for candidate in FT.fetch_orientifolds(h11=1, limit=3):
    print(candidate.yields_nef_decomposition())

print("\nOrientifolds with a nef decomposition:")
for candidate in FT.fetch_orientifolds(
    h11=1,
    limit=3,
    only_nef_decomposition=True,
):
    print(candidate.yields_nef_decomposition())

print("\nF-theory uplifts with a nef partition:")
for candidate in FT.fetch_F_Theory_uplifts(
    h11=2,
    limit=4,
    only_nef_partition=True,
):
    print(candidate.is_nef_partition())

Orientifolds:
False
False
False
True
True
True
False
True
True

Orientifolds with a nef decomposition:
True
True
True
True
True

F-theory uplifts with a nef partition:
True
True
True
True
True


## Summary

This notebook illustrated how to:

1. define and normalize a toric polytope;
2. choose or compute a $\mathbb{Z}_2$ orientifold action;
3. construct a `CY_orientifold`;
4. build the corresponding `F_Theory_Uplift`;
5. inspect fans, line bundles, polytopes, Hodge numbers, and intersection data; and
6. scan examples for nef decompositions and nef partitions.